# Paper 2, GPU stage 1: format x key/value factorial at 3-14B

Runs `experiments/format_factorial.py` (forward passes only, no training) on open 3-14B models and writes results to Google Drive (`MyDrive/causal-keys-results/`), where they can be read back for analysis.

**Runtime:** GPU (A100 40GB recommended; L4 works for <=8B). Expected roughly 5-10 min per 7B model at n=150 and 5 formats.
Gated models (Llama, Gemma) need `HF_TOKEN` in Colab secrets and accepted licences; they are skipped otherwise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/causal-keys-results'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q -b claude/paper2-research https://github.com/syedzahedi1990/Causal-keys.git /content/Causal-keys
%cd /content/Causal-keys
!pip -q install -U 'transformers>=4.57' accelerate
!git log --oneline | head -3

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception as e:
    print('No HF_TOKEN; gated models will be skipped:', type(e).__name__)

In [ ]:
import subprocess, torch, os
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODELS = ['Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 'Qwen/Qwen3-8B',
          'mistralai/Mistral-7B-Instruct-v0.3', 'allenai/OLMo-2-1124-7B-Instruct']
if gpu_gb > 35: MODELS += ['Qwen/Qwen2.5-14B-Instruct']
if os.environ.get('HF_TOKEN'): MODELS += ['meta-llama/Llama-3.1-8B-Instruct', 'google/gemma-2-9b-it']
print(MODELS)

In [ ]:
for m in MODELS:
    print('=' * 20, m, flush=True)
    r = subprocess.run(['python', 'experiments/format_factorial.py', '--model', m, '--n', '150', '--dtype', 'bfloat16',
                        '--arms', 'P1,NONE,BEFORE,POST,LETTER', '--specificity-arms', '',
                        '--out', f'{OUT}/format_factorial'],
                       env={**os.environ, 'PYTHONPATH': '.'}, capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-1500:] if r.returncode else '')

When this finishes, the summaries are in `MyDrive/causal-keys-results/format_factorial/*_summary.txt` and the raw per-item JSON next to them.